# TrashCan — Final Cleaning and Video-wise Split

**Objectif :** construire un dataset COCO propre pour la détection des 8 classes de déchets de `material_version`.

Cette version :
- retire les images contenant uniquement le ROV ;
- conserve les images négatives (aucune annotation de déchet après filtrage) ;
- conserve uniquement les 8 classes `trash_*` ;
- regroupe les images par vidéo pour éviter la fuite Train / Validation ;
- cherche un split proche de 80/20 tout en gardant chaque classe dans les deux ensembles ;
- clippe les bounding boxes et les polygones ;
- renumérote les IDs COCO ;
- vérifie les fichiers exportés.

**Important :** ce notebook crée un nouveau dossier `data/processed/trashcan_cleaned_final/` et ne modifie jamais `data/raw/`.

## 0. Imports et chemins

In [16]:
from pathlib import Path
import json
import re
import shutil
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import GroupShuffleSplit

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_PATH = PROJECT_ROOT / "data" / "raw" / "dataset" / "material_version"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "trashcan_cleaned_final"

assert DATASET_PATH.exists(), f"Dataset introuvable : {DATASET_PATH}"
print("Source:", DATASET_PATH)
print("Output:", OUTPUT_PATH)

Source: c:\Users\mariem\Documents\trashcan-dl-mlops\data\raw\dataset\material_version
Output: c:\Users\mariem\Documents\trashcan-dl-mlops\data\processed\trashcan_cleaned_final


## 1. Charger les deux annotations COCO

In [17]:
with open(DATASET_PATH / "instances_train_trashcan.json", "r", encoding="utf-8") as f:
    source_train = json.load(f)
with open(DATASET_PATH / "instances_val_trashcan.json", "r", encoding="utf-8") as f:
    source_val = json.load(f)

source_data = {"train": source_train, "val": source_val}
for split, data in source_data.items():
    print(split, "| images:", len(data["images"]), "| annotations:", len(data["annotations"]))

train | images: 6008 | annotations: 9741
val | images: 1204 | annotations: 2595


## 2. Définir les classes du modèle et analyser les images

Les images contenant uniquement `rov` sont retirées. Les images qui contiennent des animaux/plantes sans déchet peuvent rester comme exemples négatifs : le modèle doit apprendre à ne pas détecter ces objets comme des déchets.

In [ ]:
TRASH_CLASSES = [
    "trash_etc",
    "trash_plastic",
    "trash_metal",
    "trash_wood",
    "trash_fabric",
    "trash_fishing_gear",
    "trash_paper",
    "trash_rubber"
]
# IDs de catégories d'origine (identiques à ceux du notebook du partenaire)
SOURCE_CATEGORIES = [c for c in source_train["categories"] if c["name"] in TRASH_CLASSES]
CLASS_TO_ID = {c["name"]: c["id"] for c in SOURCE_CATEGORIES}
ID_TO_CLASS = {category_id: name for name, category_id in CLASS_TO_ID.items()}
assert {c["id"]: c["name"] for c in source_val["categories"] if c["name"] in TRASH_CLASSES} == ID_TO_CLASS

image_rows = []
annotation_records = []
removed_rov_only = 0

for source_split, coco in source_data.items():
    categories = {cat["id"]: cat["name"] for cat in coco["categories"]}
    image_by_id = {img["id"]: img for img in coco["images"]}

    classes_by_image = {}
    for ann in coco["annotations"]:
        classes_by_image.setdefault(ann["image_id"], set()).add(
            categories[ann["category_id"]]
        )

    rov_only_ids = {
        image_id for image_id, classes in classes_by_image.items()
        if classes == {"rov"}
    }
    removed_rov_only += len(rov_only_ids)

    for img in coco["images"]:
        if img["id"] in rov_only_ids:
            continue
        match = re.search(r"(vid_\d+)", img["file_name"])
        assert match is not None, f"video_id introuvable dans {img['file_name']}"
        image_rows.append({
            "source_split": source_split,
            "old_image_id": img["id"],
            "file_name": Path(img["file_name"]).name,
            "video_id": match.group(1),
            "width": img["width"],
            "height": img["height"],
            "image_info": img
        })

    for ann in coco["annotations"]:
        class_name = categories[ann["category_id"]]
        if class_name not in TRASH_CLASSES:
            continue
        if ann["image_id"] in rov_only_ids:
            continue
        annotation_records.append({
            "source_split": source_split,
            "old_image_id": ann["image_id"],
            "class_name": class_name,
            "annotation": ann
        })

images_df = pd.DataFrame(image_rows)
annotations_df = pd.DataFrame(annotation_records)

print("ROV-only images removed:", removed_rov_only)
print("Images retained:", len(images_df))
print("Trash annotations retained:", len(annotations_df))
print("Unique videos:", images_df["video_id"].nunique())
print("Images without any trash annotation (negative images):",
      len(images_df) - annotations_df[["source_split", "old_image_id"]].drop_duplicates().shape[0])

assert set(annotations_df["class_name"].unique()) == set(TRASH_CLASSES),     "Une ou plusieurs classes de déchets sont absentes du dataset source."
assert images_df["file_name"].is_unique, "Noms de fichiers dupliqués entre les deux splits."

ROV-only images removed: 730
Images retained: 6482
Trash annotations retained: 6214
Unique videos: 303
Images without any trash annotation (negative images): 1260


## 3. Préparer les statistiques par vidéo

In [19]:
# Clé unique d'une image source : le même ancien image_id peut exister dans les deux JSON.
image_key_to_video = {
    (row.source_split, row.old_image_id): row.video_id
    for row in images_df.itertuples()
}

annotations_df["video_id"] = annotations_df.apply(
    lambda row: image_key_to_video[(row["source_split"], row["old_image_id"])],
    axis=1
)

video_class_counts = pd.crosstab(
    annotations_df["video_id"],
    annotations_df["class_name"]
).reindex(columns=TRASH_CLASSES, fill_value=0)

total_class_counts = video_class_counts.sum(axis=0)
display(total_class_counts.to_frame("Total annotations"))
display(
    annotations_df.groupby("class_name")["video_id"]
    .nunique().reindex(TRASH_CLASSES)
    .to_frame("Videos containing class")
)

,Total annotations
class_name,
trash_etc,2039
trash_plastic,1864
trash_metal,1126
trash_wood,340
trash_fabric,314
trash_fishing_gear,196
trash_paper,193
trash_rubber,142


,Videos containing class
class_name,
trash_etc,154
trash_plastic,97
trash_metal,62
trash_wood,20
trash_fabric,21
trash_fishing_gear,14
trash_paper,10
trash_rubber,7


## 4. Trouver un split Train / Validation par vidéo

On teste plusieurs séparations groupées. Une vidéo complète va soit dans Train, soit dans Validation. Le score cherche à équilibrer les proportions des 8 classes et à garder une validation proche de 20 % des images.

In [20]:
best_score = float("inf")
best_train_idx = None
best_val_idx = None
TARGET_VAL_RATIO = 0.20
N_CANDIDATES = 3000

splitter = GroupShuffleSplit(
    n_splits=N_CANDIDATES,
    test_size=TARGET_VAL_RATIO,
    random_state=42
)

groups = images_df["video_id"].to_numpy()
dummy_x = np.zeros((len(images_df), 1))

for train_idx, val_idx in splitter.split(dummy_x, groups=groups):
    val_video_ids = set(images_df.iloc[val_idx]["video_id"])
    val_counts = video_class_counts.loc[
        video_class_counts.index.isin(val_video_ids)
    ].sum(axis=0)
    train_counts = total_class_counts - val_counts

    # Toutes les classes doivent être représentées dans les deux ensembles.
    if (val_counts == 0).any() or (train_counts == 0).any():
        continue

    class_ratios = (val_counts / total_class_counts).to_numpy()
    val_image_ratio = len(val_idx) / len(images_df)

    class_error = np.mean((class_ratios - TARGET_VAL_RATIO) ** 2)
    image_error = (val_image_ratio - TARGET_VAL_RATIO) ** 2
    score = class_error + 2.0 * image_error

    if score < best_score:
        best_score = score
        best_train_idx = train_idx
        best_val_idx = val_idx

assert best_train_idx is not None, (
    "Aucun split valide trouvé. Vérifie la présence des classes par vidéo."
)

train_rows = images_df.iloc[best_train_idx].copy().reset_index(drop=True)
val_rows = images_df.iloc[best_val_idx].copy().reset_index(drop=True)

train_videos = set(train_rows["video_id"])
val_videos = set(val_rows["video_id"])

print("Train images:", len(train_rows))
print("Validation images:", len(val_rows))
print("Train ratio:", round(len(train_rows) / len(images_df) * 100, 2), "%")
print("Validation ratio:", round(len(val_rows) / len(images_df) * 100, 2), "%")
print("Train videos:", len(train_videos))
print("Validation videos:", len(val_videos))
print("Shared videos:", len(train_videos & val_videos))
print("Split score:", best_score)

assert train_videos.isdisjoint(val_videos)

Train images: 5233
Validation images: 1249
Train ratio: 80.73 %
Validation ratio: 19.27 %
Train videos: 242
Validation videos: 61
Shared videos: 0
Split score: 0.0013520781251139466


## 5. Vérifier la distribution des classes après le split

In [21]:
val_video_ids = set(val_rows["video_id"])
val_counts = video_class_counts.loc[
    video_class_counts.index.isin(val_video_ids)
].sum(axis=0)
train_counts = total_class_counts - val_counts

distribution = pd.DataFrame({
    "Train": train_counts,
    "Validation": val_counts,
    "Total": total_class_counts
}).reindex(TRASH_CLASSES)
distribution["Validation (%)"] = (
    100 * distribution["Validation"] / distribution["Total"]
).round(2)
display(distribution)

assert (distribution["Train"] > 0).all()
assert (distribution["Validation"] > 0).all()

,Train,Validation,Total,Validation (%)
class_name,,,,
trash_etc,1589,450,2039,22.07
trash_plastic,1467,397,1864,21.30
trash_metal,908,218,1126,19.36
trash_wood,285,55,340,16.18
trash_fabric,258,56,314,17.83
trash_fishing_gear,160,36,196,18.37
trash_paper,168,25,193,12.95
trash_rubber,107,35,142,24.65


## 6. Fonctions de correction des coordonnées

Les coordonnées sont limitées aux dimensions de l'image. Les polygones COCO sont également découpés aux limites de l'image. Les annotations ne sont pas supprimées simplement parce qu'elles dépassent légèrement les bords.

In [22]:
def polygon_area(points):
    if len(points) < 6:
        return 0.0
    xs = points[0::2]
    ys = points[1::2]
    total = 0.0
    for i in range(len(xs)):
        j = (i + 1) % len(xs)
        total += xs[i] * ys[j] - xs[j] * ys[i]
    return abs(total) / 2.0


def clip_polygon(points, width, height):
    if len(points) < 6 or len(points) % 2 != 0:
        return []

    polygon = [(float(points[i]), float(points[i + 1]))
               for i in range(0, len(points), 2)]

    # Sutherland-Hodgman clipping against x>=0, x<=width, y>=0, y<=height.
    boundaries = [
        ("x_min", 0.0), ("x_max", float(width)),
        ("y_min", 0.0), ("y_max", float(height))
    ]

    for boundary, value in boundaries:
        if not polygon:
            break

        output = []
        previous = polygon[-1]

        def inside(point):
            x, y = point
            if boundary == "x_min": return x >= value
            if boundary == "x_max": return x <= value
            if boundary == "y_min": return y >= value
            return y <= value

        def intersection(a, b):
            x1, y1 = a
            x2, y2 = b
            if boundary in ("x_min", "x_max"):
                if x2 == x1:
                    return (value, y1)
                t = (value - x1) / (x2 - x1)
                return (value, y1 + t * (y2 - y1))
            if y2 == y1:
                return (x1, value)
            t = (value - y1) / (y2 - y1)
            return (x1 + t * (x2 - x1), value)

        for current in polygon:
            current_inside = inside(current)
            previous_inside = inside(previous)

            if current_inside:
                if not previous_inside:
                    output.append(intersection(previous, current))
                output.append(current)
            elif previous_inside:
                output.append(intersection(previous, current))

            previous = current

        polygon = output

    flattened = []
    for x, y in polygon:
        flattened.extend([min(max(x, 0.0), float(width)),
                          min(max(y, 0.0), float(height))])
    return flattened


def clean_annotation(source_ann, new_image_id, new_ann_id, class_name, width, height):
    ann = dict(source_ann)

    x, y, box_w, box_h = map(float, ann["bbox"])
    x1 = min(max(x, 0.0), float(width))
    y1 = min(max(y, 0.0), float(height))
    x2 = min(max(x + box_w, 0.0), float(width))
    y2 = min(max(y + box_h, 0.0), float(height))

    if x2 <= x1 or y2 <= y1:
        raise ValueError(
            f"Bounding box vide après clipping (annotation id={source_ann.get('id')})."
        )

    new_bbox = [x1, y1, x2 - x1, y2 - y1]

    segmentation = ann.get("segmentation", [])
    new_segmentation = segmentation
    area = new_bbox[2] * new_bbox[3]

    if isinstance(segmentation, list):
        clipped_polygons = []
        polygon_areas = []
        for polygon in segmentation:
            clipped = clip_polygon(polygon, width, height)
            if len(clipped) >= 6:
                clipped_polygons.append(clipped)
                polygon_areas.append(polygon_area(clipped))
        new_segmentation = clipped_polygons
        if polygon_areas:
            area = float(sum(polygon_areas))

    ann.update({
        "id": new_ann_id,
        "image_id": new_image_id,
        "category_id": CLASS_TO_ID[class_name],
        "bbox": new_bbox,
        "area": float(area),
        "segmentation": new_segmentation,
        "iscrowd": int(ann.get("iscrowd", 0))
    })
    return ann

## 7. Reconstruire les annotations COCO avec des IDs propres

In [23]:
def build_coco(split_rows, split_name):
    final_images = []
    final_annotations = []

    # IDs d'image nouveaux et continus à partir de 1.
    source_to_new_id = {}
    for new_id, row in enumerate(split_rows.itertuples(), start=1):
        source_key = (row.source_split, row.old_image_id)
        source_to_new_id[source_key] = new_id

        image_info = dict(row.image_info)
        image_info["id"] = new_id
        image_info["file_name"] = Path(image_info["file_name"]).name
        final_images.append(image_info)

    # Index des dimensions pour le clipping.
    dimensions = {
        (row.source_split, row.old_image_id): (int(row.width), int(row.height))
        for row in split_rows.itertuples()
    }

    split_keys = set(source_to_new_id.keys())
    ann_id = 1

    for record in annotations_df.to_dict("records"):
        source_key = (record["source_split"], record["old_image_id"])
        if source_key not in split_keys:
            continue

        width, height = dimensions[source_key]
        cleaned_ann = clean_annotation(
            record["annotation"],
            source_to_new_id[source_key],
            ann_id,
            record["class_name"],
            width,
            height
        )
        final_annotations.append(cleaned_ann)
        ann_id += 1

    categories = [
        {"id": CLASS_TO_ID[name], "name": name, "supercategory": "trash"}
        for name in TRASH_CLASSES
    ]

    return {
        "images": final_images,
        "annotations": final_annotations,
        "categories": categories
    }


train_coco = build_coco(train_rows, "train")
val_coco = build_coco(val_rows, "val")

print("Train:", len(train_coco["images"]), "images |",
      len(train_coco["annotations"]), "annotations")
print("Validation:", len(val_coco["images"]), "images |",
      len(val_coco["annotations"]), "annotations")
print("Category IDs:", {c["name"]: c["id"] for c in train_coco["categories"]})

Train: 5233 images | 4942 annotations
Validation: 1249 images | 1272 annotations
Category IDs: {'trash_etc': 1, 'trash_plastic': 2, 'trash_metal': 3, 'trash_wood': 4, 'trash_fabric': 5, 'trash_fishing_gear': 6, 'trash_paper': 7, 'trash_rubber': 8}


## 8. Vérifications avant export

In [24]:
def validate_coco(coco, split_name):
    image_ids = [img["id"] for img in coco["images"]]
    ann_ids = [ann["id"] for ann in coco["annotations"]]
    image_id_set = set(image_ids)

    assert len(image_ids) == len(set(image_ids)), f"{split_name}: image IDs dupliqués"
    assert len(ann_ids) == len(set(ann_ids)), f"{split_name}: annotation IDs dupliqués"
    assert all(ann["image_id"] in image_id_set for ann in coco["annotations"])
    assert {cat["id"] for cat in coco["categories"]} == set(range(1, 9))
    assert {cat["name"] for cat in coco["categories"]} == set(TRASH_CLASSES)

    image_lookup = {img["id"]: img for img in coco["images"]}
    for ann in coco["annotations"]:
        img = image_lookup[ann["image_id"]]
        x, y, w, h = ann["bbox"]
        assert w > 0 and h > 0
        assert x >= -1e-6 and y >= -1e-6
        assert x + w <= img["width"] + 1e-6
        assert y + h <= img["height"] + 1e-6
        assert ann["category_id"] in range(1, 9)

    counts = Counter(ann["category_id"] for ann in coco["annotations"])
    assert set(counts.keys()) == set(range(1, 9)), f"{split_name}: classe absente"
    print(
        split_name,
        "| images:", len(image_ids),
        "| annotations:", len(ann_ids),
        "| negative images:", len(image_ids) - len({a["image_id"] for a in coco["annotations"]}),
        "| all 8 classes present: OK"
    )

validate_coco(train_coco, "TRAIN")
validate_coco(val_coco, "VALIDATION")
assert train_videos.isdisjoint(val_videos), "Une vidéo apparaît dans les deux splits."
print("Video leakage check: OK")

TRAIN | images: 5233 | annotations: 4942 | negative images: 1058 | all 8 classes present: OK
VALIDATION | images: 1249 | annotations: 1272 | negative images: 202 | all 8 classes present: OK
Video leakage check: OK


## 9. Exporter les images et les annotations

Le dossier `trashcan_cleaned_final` est dédié à ce notebook. À chaque exécution, il est recréé pour éviter de conserver des images d'une ancienne exécution.

In [25]:
# Attention : seul le dossier de sortie dédié de ce notebook est supprimé/recréé.
if OUTPUT_PATH.exists():
    shutil.rmtree(OUTPUT_PATH)

(OUTPUT_PATH / "train" / "images").mkdir(parents=True, exist_ok=True)
(OUTPUT_PATH / "val" / "images").mkdir(parents=True, exist_ok=True)
(OUTPUT_PATH / "annotations").mkdir(parents=True, exist_ok=True)


def copy_split_images(split_rows, split_name):
    destination = OUTPUT_PATH / split_name / "images"
    for row in split_rows.itertuples():
        source = DATASET_PATH / row.source_split / Path(row.file_name).name
        assert source.is_file(), f"Image source manquante : {source}"
        shutil.copy2(source, destination / Path(row.file_name).name)
    print(split_name, "images copied:", len(split_rows))


copy_split_images(train_rows, "train")
copy_split_images(val_rows, "val")

with open(OUTPUT_PATH / "annotations" / "instances_train.json", "w", encoding="utf-8") as f:
    json.dump(train_coco, f, indent=2)
with open(OUTPUT_PATH / "annotations" / "instances_val.json", "w", encoding="utf-8") as f:
    json.dump(val_coco, f, indent=2)

with open(OUTPUT_PATH / "annotations" / "categories.json", "w", encoding="utf-8") as f:
    json.dump(train_coco["categories"], f, indent=2)

print("Export completed:", OUTPUT_PATH)

train images copied: 5233
val images copied: 1249
Export completed: c:\Users\mariem\Documents\trashcan-dl-mlops\data\processed\trashcan_cleaned_final


In [26]:
import json
from pathlib import Path

OUTPUT_PATH = Path("C:/Users/mariem/Documents/trashcan-dl-mlops/data/processed/trashcan_cleaned_final")

for split in ["train", "val"]:

    json_path = OUTPUT_PATH / "annotations" / f"instances_{split}.json"

    with open(json_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    image_lookup = {
        img["id"]: img
        for img in coco["images"]
    }

    invalid_polygons = 0
    checked_polygons = 0

    for ann in coco["annotations"]:

        image = image_lookup[ann["image_id"]]
        width = image["width"]
        height = image["height"]

        for polygon in ann.get("segmentation", []):

            if len(polygon) < 6:
                continue

            checked_polygons += 1

            for i in range(0, len(polygon), 2):

                x = polygon[i]
                y = polygon[i + 1]

                if not (0 <= x <= width and 0 <= y <= height):
                    invalid_polygons += 1

    print(f"{split.upper()}:")
    print("Polygons checked:", checked_polygons)
    print("Invalid polygons:", invalid_polygons)

    assert invalid_polygons == 0

print("Polygon validation completed successfully!")

TRAIN:
Polygons checked: 7117
Invalid polygons: 0
VAL:
Polygons checked: 1822
Invalid polygons: 0
Polygon validation completed successfully!


In [28]:
import pandas as pd

ANNOTATIONS_PATH = OUTPUT_PATH / "annotations"


def export_to_csv(coco, split_name):
    # Images
    images_out = pd.DataFrame(coco["images"])
    images_out.to_csv(
        ANNOTATIONS_PATH / f"{split_name}_images.csv",
        index=False, encoding="utf-8-sig"
    )

    # Annotations : bbox [x, y, w, h] -> 4 colonnes
    annotations_out = pd.DataFrame(coco["annotations"])
    annotations_out[["bbox_x", "bbox_y", "bbox_width", "bbox_height"]] = pd.DataFrame(
        annotations_out["bbox"].tolist(), index=annotations_out.index
    )
    annotations_out = annotations_out.drop(columns=["bbox"])
    annotations_out.to_csv(
        ANNOTATIONS_PATH / f"{split_name}_annotations.csv",
        index=False, encoding="utf-8-sig"
    )

    print(f"{split_name.upper()} CSV | images: {len(images_out)} | annotations: {len(annotations_out)}")


export_to_csv(train_coco, "train")
export_to_csv(val_coco, "val")

pd.DataFrame(train_coco["categories"]).to_csv(
    ANNOTATIONS_PATH / "categories.csv", index=False, encoding="utf-8-sig"
)
print("Fichiers dans annotations/ :", sorted(p.name for p in ANNOTATIONS_PATH.iterdir()))

TRAIN CSV | images: 5233 | annotations: 4942
VAL CSV | images: 1249 | annotations: 1272
Fichiers dans annotations/ : ['categories.csv', 'categories.json', 'instances_train.json', 'instances_val.json', 'train_annotations.csv', 'train_images.csv', 'val_annotations.csv', 'val_images.csv']


## 10. Vérification des fichiers exportés

In [27]:
def verify_export(coco, split_name):
    images_dir = OUTPUT_PATH / split_name / "images"
    json_names = {img["file_name"] for img in coco["images"]}
    disk_names = {p.name for p in images_dir.iterdir() if p.is_file()}

    missing = json_names - disk_names
    extra = disk_names - json_names

    print(f"{split_name}: JSON={len(json_names)}, disk={len(disk_names)}, missing={len(missing)}, extra={len(extra)}")
    assert not missing, f"Images manquantes dans {split_name}: {list(missing)[:5]}"
    assert not extra, f"Images supplémentaires dans {split_name}: {list(extra)[:5]}"


verify_export(train_coco, "train")
verify_export(val_coco, "val")

# Résumé des annotations finales
final_distribution = pd.DataFrame({
    "Train": pd.Series(Counter(a["category_id"] for a in train_coco["annotations"])),
    "Validation": pd.Series(Counter(a["category_id"] for a in val_coco["annotations"]))
}).fillna(0).astype(int).sort_index()
final_distribution.index = TRASH_CLASSES
display(final_distribution)

print("FINAL DATASET READY:", OUTPUT_PATH)

train: JSON=5233, disk=5233, missing=0, extra=0
val: JSON=1249, disk=1249, missing=0, extra=0


,Train,Validation
trash_etc,1589,450
trash_plastic,1467,397
trash_metal,908,218
trash_wood,285,55
trash_fabric,258,56
trash_fishing_gear,160,36
trash_paper,168,25
trash_rubber,107,35


FINAL DATASET READY: C:\Users\mariem\Documents\trashcan-dl-mlops\data\processed\trashcan_cleaned_final


## Résultat attendu et suite du projet

Structure finale :

```text
data/
└── processed/
    └── trashcan_cleaned_final/
        ├── train/
        │   └── images/
        ├── val/
        │   └── images/
        └── annotations/
            ├── instances_train.json
            ├── instances_val.json
            └── categories.json
```

**Convention des IDs :** les catégories sont numérotées de 1 à 8. C'est adapté à un détecteur `torchvision` qui réserve la classe `0` au background. Si tu choisis YOLO plus tard, la conversion COCO vers YOLO devra transformer ces IDs en 0 à 7.

Ne relance pas l'EDA après le Cleaning : l'EDA décrit le dataset original. Une fois ce notebook exécuté et les vérifications réussies, l'étape suivante sera de visualiser un échantillon du dataset nettoyé puis de préparer le `Dataset` / `DataLoader` du modèle.